# 05_Statistical_Analysis

## Purpose

This notebook performs the statistical analyses reported in the manuscript, including bootstrap confidence intervals for accuracy and McNemar's exact test for pairwise model comparisons.

## Prerequisites

Execute `04_Model_Evaluation.ipynb` first (or load the saved prediction arrays) so that the following arrays are available:

- `y_true`, `y_pred`
- `hy_true`, `hy_pred`
- `vit_true`, `vit_pred`
- `swin_true`, `swin_pred`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score
from statsmodels.stats.contingency_tables import mcnemar
from IPython.display import display

BASE_DIR = "/content/drive/MyDrive/NAVARASA_PROJECT_DRIVE1"
RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)


## Bootstrap Confidence Interval

In [ ]:
def bootstrap_accuracy_ci(
    y_true,
    y_pred,
    n_bootstrap=2000,
    confidence=95,
    seed=42
):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    scores = []
    n = len(y_true)

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        scores.append(accuracy_score(y_true[idx], y_pred[idx]))

    lower = np.percentile(scores, (100-confidence)/2)
    upper = np.percentile(scores, 100-(100-confidence)/2)
    return lower, upper


In [ ]:
accuracy = accuracy_score(y_true, y_pred)

lower, upper = bootstrap_accuracy_ci(y_true, y_pred)

print("="*60)
print("95% Confidence Interval")
print("="*60)
print(f"Accuracy : {accuracy:.4f}")
print(f"95% CI   : [{lower:.4f}, {upper:.4f}]")

bootstrap_df = pd.DataFrame({
    "Metric":["Accuracy"],
    "Accuracy":[accuracy],
    "CI Lower":[lower],
    "CI Upper":[upper],
    "Confidence Level":["95%"],
    "Bootstrap Samples":[2000]
})

bootstrap_path = os.path.join(
    RESULTS_DIR,
    "Bootstrap_Confidence_Interval.csv"
)

bootstrap_df.to_csv(bootstrap_path, index=False)
print(f"Bootstrap results saved to: {bootstrap_path}")


## McNemar Exact Test

In [ ]:
def run_mcnemar(y_true, predA, predB):
    correctA = (predA == y_true)
    correctB = (predB == y_true)

    b = np.sum(correctA & (~correctB))
    c = np.sum((~correctA) & correctB)

    table = [[0, b],
             [c, 0]]

    result = mcnemar(table, exact=True)
    return b, c, result.pvalue


In [ ]:
b1, c1, p1 = run_mcnemar(hy_true, hy_pred, vit_pred)
b2, c2, p2 = run_mcnemar(hy_true, hy_pred, swin_pred)
b3, c3, p3 = run_mcnemar(vit_true, vit_pred, swin_pred)

mcnemar_table = pd.DataFrame({
    "Comparison": [
        "Hybrid vs ViT",
        "Hybrid vs Swin",
        "ViT vs Swin"
    ],
    "b": [b1, b2, b3],
    "c": [c1, c2, c3],
    "Exact p-value": [p1, p2, p3]
})

mcnemar_table["Exact p-value"] = mcnemar_table["Exact p-value"].apply(lambda x: f"{x:.2e}")

mcnemar_table["Significant (α = 0.05)"] = [
    "Yes" if p1 < 0.05 else "No",
    "Yes" if p2 < 0.05 else "No",
    "Yes" if p3 < 0.05 else "No"
]

mcnemar_table["Better Model"] = [
    "Hybrid",
    "Hybrid",
    "ViT"
]

display(mcnemar_table)

mcnemar_path = os.path.join(RESULTS_DIR, "Table_McNemar_Test.csv")
mcnemar_table.to_csv(mcnemar_path, index=False)

print(f"McNemar table saved to: {mcnemar_path}")
